# Dog vs Cat Classification with a 3-Conv-Layer CNN
Label convention used everywhere: **0 = Cat, 1 = Dog**. The sigmoid output is P(Dog).

## Setup (Google Colab)
1. Upload the whole `dog_cat_classification` folder (with `dataset/cats` and `dataset/dogs` filled) to Colab or Google Drive.
2. In Colab, run `%cd /content/dog_cat_classification/notebooks` in a code cell before running the cells below.
3. Set `PER_CLASS` in the first code cell (e.g. 500 for a quick test, 2000 for the project run).

## 1. Imports and settings
(Colab users: run `!pip install -q tensorflow scikit-learn pillow matplotlib` first if anything is missing.)

In [ ]:
import hashlib
import json
import os
import random
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PIL import Image, UnidentifiedImageError
from sklearn.metrics import (ConfusionMatrixDisplay, classification_report,
                             confusion_matrix, precision_recall_fscore_support)
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

SEED = 42
IMG_SIZE = 128
BATCH_SIZE = 32
MAX_EPOCHS = 20
PER_CLASS = int(os.environ.get("PER_CLASS", 2000))  # images used per class (use 500 for a quick test)
CLASS_NAMES = ["Cat", "Dog"]                        # index 0 = Cat, index 1 = Dog
EXTENSIONS = {".jpg", ".jpeg", ".png"}

try:
    ROOT = Path(__file__).resolve().parent.parent   # running as a script (src/train_model.py)
except NameError:                                   # running inside a notebook
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "dataset"
OUT_DIR = ROOT / "outputs"
MODEL_PATH = ROOT / "dog_cat_cnn.keras"
OUT_DIR.mkdir(exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


def save_fig(name):
    """Save the current figure into outputs/ and show it when running in a notebook."""
    plt.tight_layout()
    plt.savefig(OUT_DIR / name, dpi=150)
    if matplotlib.get_backend().lower() == "agg":
        plt.close()
    else:
        plt.show()


def load_and_resize(path):
    """Open an image, force RGB, resize to 128x128. Returns a uint8 array. Used for training AND prediction."""
    with Image.open(path) as img:
        return np.array(img.convert("RGB").resize((IMG_SIZE, IMG_SIZE)), dtype=np.uint8)

## 2. Collect image paths, remove corrupted files and exact duplicates
Duplicates are found by hashing the file bytes (MD5). Removing them *before* splitting
guarantees the same image can never appear in two different splits (no data leakage).

In [ ]:
def collect_files(folder):
    folder = Path(folder)
    if not folder.exists():
        raise FileNotFoundError(f"Folder not found: {folder}\nSee README.md for how to set up the dataset.")
    return sorted(p for p in folder.iterdir() if p.suffix.lower() in EXTENSIONS)


paths, labels, seen_hashes = [], [], set()
skipped_corrupt = skipped_duplicate = 0

for label, sub in enumerate(["cats", "dogs"]):
    kept = 0
    for p in collect_files(DATA_DIR / sub):
        if kept >= PER_CLASS:
            break
        try:
            with Image.open(p) as img:
                img.verify()
        except (UnidentifiedImageError, OSError, SyntaxError):
            skipped_corrupt += 1
            continue
        digest = hashlib.md5(p.read_bytes()).hexdigest()
        if digest in seen_hashes:
            skipped_duplicate += 1
            continue
        seen_hashes.add(digest)
        paths.append(str(p))
        labels.append(label)
        kept += 1

labels = np.array(labels)
print(f"Images kept: {len(paths)}  (cats={int((labels == 0).sum())}, dogs={int((labels == 1).sum())})")
print(f"Skipped corrupted: {skipped_corrupt} | Skipped duplicates: {skipped_duplicate}")
if len(paths) < 20 or len(set(labels)) < 2:
    raise RuntimeError("Not enough images found. Put images in dataset/cats and dataset/dogs (see README.md).")

## 3. Split 70% train / 15% validation / 15% test
We split the *file list* (not pixels). `stratify` keeps the cat/dog ratio equal in every split and
`random_state=SEED` makes the split reproducible.

In [ ]:
p_train, p_temp, y_train, y_temp = train_test_split(
    paths, labels, test_size=0.30, stratify=labels, random_state=SEED)
p_val, p_test, y_val, y_test = train_test_split(
    p_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)

assert not (set(p_train) & set(p_val)) and not (set(p_train) & set(p_test)) and not (set(p_val) & set(p_test))
print(f"Train={len(p_train)}  Validation={len(p_val)}  Test={len(p_test)}  (no overlap between splits)")

## 4. Load images (resize to 128x128) and normalize to 0-1
Images are stored as small uint8 arrays to save memory and divided by 255 batch by batch.

In [ ]:
def load_array(path_list):
    return np.stack([load_and_resize(p) for p in path_list])


X_train, X_val, X_test = load_array(p_train), load_array(p_val), load_array(p_test)
print("Array shapes:", X_train.shape, X_val.shape, X_test.shape)


def make_dataset(X, y, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y.astype("float32")))
    if shuffle:
        ds = ds.shuffle(len(X), seed=SEED)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.map(lambda x, t: (tf.cast(x, tf.float32) / 255.0, t))  # normalize to 0-1
    return ds.prefetch(tf.data.AUTOTUNE)


train_ds = make_dataset(X_train, y_train, shuffle=True)
val_ds = make_dataset(X_val, y_val)
test_ds = make_dataset(X_test, y_test)          # not shuffled, so predictions line up with y_test

## 5. Show sample images with labels

In [ ]:
rng = np.random.default_rng(SEED)
idx = rng.choice(len(X_train), size=8, replace=False)
plt.figure(figsize=(12, 6))
for i, k in enumerate(idx):
    plt.subplot(2, 4, i + 1)
    plt.imshow(X_train[k])
    plt.title(f"{CLASS_NAMES[y_train[k]]} (label={y_train[k]})")
    plt.axis("off")
save_fig("sample_images.png")

## 6. Build the CNN (exactly three Conv2D layers)

In [ ]:
model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.Conv2D(32, (3, 3), activation="relu", name="conv1"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3, 3), activation="relu", name="conv2"),
    layers.MaxPooling2D(),
    layers.Conv2D(128, (3, 3), activation="relu", name="conv3"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid"),       # P(Dog); 0 = Cat, 1 = Dog
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

n_conv = sum(isinstance(l, layers.Conv2D) for l in model.layers)
assert n_conv == 3, f"Expected exactly 3 Conv2D layers, found {n_conv}"
print(f"Verified: the model has exactly {n_conv} Conv2D layers.")
with open(OUT_DIR / "model_summary.txt", "w", encoding="utf-8") as f:
    model.summary(print_fn=lambda line: f.write(line + "\n"))

## 7. Train
* **EarlyStopping** stops when validation loss stops improving for 3 epochs and restores the best weights.
* **ModelCheckpoint** saves the best model (lowest validation loss) to `dog_cat_cnn.keras`.

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    ModelCheckpoint(str(MODEL_PATH), monitor="val_loss", save_best_only=True),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=MAX_EPOCHS, callbacks=callbacks,
                    shuffle=False, verbose=2)  # data is already shuffled inside train_ds
print(f"Best model saved to: {MODEL_PATH}")

## 8. Training graphs

In [ ]:
for metric, title in [("accuracy", "Accuracy"), ("loss", "Loss")]:
    plt.figure(figsize=(6, 4))
    plt.plot(history.history[metric], label=f"Training {metric}")
    plt.plot(history.history[f"val_{metric}"], label=f"Validation {metric}")
    plt.xlabel("Epoch")
    plt.ylabel(title)
    plt.title(f"Training and Validation {title}")
    plt.legend()
    save_fig(f"{metric}_graph.png")

## 9. Evaluate on the test set (all numbers come from the real model)

In [ ]:
best_model = tf.keras.models.load_model(MODEL_PATH)
test_loss, test_acc = best_model.evaluate(test_ds, verbose=0)
probs = best_model.predict(test_ds, verbose=0).ravel()      # P(Dog)
y_pred = (probs > 0.5).astype(int)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_test, y_pred, average="binary", pos_label=1, zero_division=0)
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=CLASS_NAMES, zero_division=0)

print(f"Test loss     : {test_loss:.4f}")
print(f"Test accuracy : {test_acc:.4f}")
print(f"Precision (Dog as positive class): {precision:.4f}")
print(f"Recall    (Dog as positive class): {recall:.4f}")
print(f"F1-score  (Dog as positive class): {f1:.4f}")
print("\nConfusion matrix (rows = true, columns = predicted):\n", cm)
print("\nClassification report:\n", report)

ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(cmap="Blues")
plt.title("Confusion Matrix (Test Set)")
save_fig("confusion_matrix.png")

metrics = {"test_loss": float(test_loss), "test_accuracy": float(test_acc), "precision": float(precision),
           "recall": float(recall), "f1_score": float(f1), "confusion_matrix": cm.tolist(),
           "train_images": len(p_train), "val_images": len(p_val), "test_images": len(p_test),
           "epochs_run": len(history.history["loss"])}
(OUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
(OUT_DIR / "classification_report.txt").write_text(report)
print("Saved metrics, report and graphs to:", OUT_DIR)

## 10. Prediction on a new image

In [ ]:
def predict_image(image_path, model_to_use=None, show=True):
    """Return (class_name, confidence) for one image, or None if the file cannot be used."""
    model_to_use = model_to_use or best_model
    try:
        arr = load_and_resize(image_path)
    except FileNotFoundError:
        print(f"File not found: {image_path}")
        return None
    except (UnidentifiedImageError, OSError):
        print(f"Unsupported or corrupted image: {image_path}")
        return None
    p_dog = float(model_to_use.predict(arr[np.newaxis] / 255.0, verbose=0)[0][0])
    name = CLASS_NAMES[int(p_dog > 0.5)]
    confidence = p_dog if p_dog > 0.5 else 1 - p_dog
    if show:
        plt.figure(figsize=(3, 3))
        plt.imshow(arr)
        plt.title(f"{name} ({confidence * 100:.1f}%)")
        plt.axis("off")
        save_fig(f"prediction_{name.lower()}_{Path(image_path).stem}.png")
    print(f"{Path(image_path).name}: {name} with {confidence * 100:.1f}% confidence")
    return name, confidence


sample_cat = next(p for p, y in zip(p_test, y_test) if y == 0)
sample_dog = next(p for p, y in zip(p_test, y_test) if y == 1)
predict_image(sample_cat)
predict_image(sample_dog)
predict_image("this/path/does/not/exist.jpg")        # shows graceful error handling

## 11. Reload the saved model and predict again

In [ ]:
reloaded = tf.keras.models.load_model(MODEL_PATH)
print("Reloaded model from:", MODEL_PATH)
predict_image(sample_cat, reloaded)
predict_image(sample_dog, reloaded)